In [1]:
import logging
import pandas as pd
import numpy as np
import os

logging.basicConfig(level=logging.INFO)

class FileValidationError(Exception):
    pass

class FileMissingValueError(FileValidationError):
    pass

def load_results(filepath: str) -> pd.DataFrame:
    if not os.path.exists(filepath):
        raise FileValidationError("File not found!!")  
    elif os.path.getsize(filepath) == 0:
        raise FileMissingValueError("File exists but contains no value or data")
    else:
        df = pd.read_csv(filepath, encoding='utf-8-sig')
        logging.info(f"Loaded {filepath} — shape: {df.shape}")
        return df

try:
    al = load_results("data/raw/Allocated Limit for Honble MPs.csv")
    wc = load_results("data/raw/Works Completed.csv")
    ws = load_results("data/raw/Works Sanctioned.csv")

except FileValidationError as e:
    logging.error(f"Loading Failed: {e}")
    raise

INFO:root:Loaded data/raw/Allocated Limit for Honble MPs.csv — shape: (544, 5)
INFO:root:Loaded data/raw/Works Completed.csv — shape: (15001, 11)
INFO:root:Loaded data/raw/Works Sanctioned.csv — shape: (6001, 12)


In [2]:
import re

def clean_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    
    dfCols = list(df.columns)
    columns_cleaned = []

    for col in dfCols:
        cleaned_names = re.sub("[\.\'\()\₹]", '',str(col)).lower().strip()
        cleaned_names = cleaned_names.replace(" ", "_")
        columns_cleaned.append(cleaned_names)

    df.columns = (columns_cleaned)
    return df


clean_dataframe_columns(al).columns
clean_dataframe_columns(wc).columns
clean_dataframe_columns(ws).columns

Index(['sr_no', 'work_category', 'work', 'state', 'ida',
       'honble_members_of_parliament', 'constituency', 'work_description',
       'recommended_date', 'sanction_date', 'sanction_amount', 'work_status'],
      dtype='object')

In [3]:
al.columns, ws.columns, wc.columns

(Index(['sr_no', 'state', 'honble_members_of_parliaments', 'constituency',
        'allocated_amount'],
       dtype='object'),
 Index(['sr_no', 'work_category', 'work', 'state', 'ida',
        'honble_members_of_parliament', 'constituency', 'work_description',
        'recommended_date', 'sanction_date', 'sanction_amount', 'work_status'],
       dtype='object'),
 Index(['sr_no', 'work_category', 'work', 'state', 'ida', 'work_description',
        'honble_members_of_parliament', 'constituency', 'image',
        'completion_date', 'amount_disbursed'],
       dtype='object'))

In [4]:
def diagnose_dataframe(df: pd.DataFrame, name: str):
    logging.info(f"\n Dataframe -> {name}\n")
    logging.info(f" Shape: \n{df.shape}\n")
    logging.info(f" Dataframe Datatypes \n{df.dtypes}\n")
    logging.info(f" Dataframe null columns \n{df.isna().sum()[df.isna().sum()>0]}\n")
    logging.info(f" Datarframe duplicates \n{df.duplicated().sum()}")

    
diagnose_dataframe(al, "Allocated Limit for Honble MPs")
diagnose_dataframe(wc, "Works Completed")
diagnose_dataframe(ws, "Works Sanctioned")

INFO:root:
 Dataframe -> Allocated Limit for Honble MPs

INFO:root: Shape: 
(544, 5)

INFO:root: Dataframe Datatypes 
sr_no                            object
state                            object
honble_members_of_parliaments    object
constituency                     object
allocated_amount                 object
dtype: object

INFO:root: Dataframe null columns 
allocated_amount    1
dtype: int64

INFO:root: Datarframe duplicates 
0
INFO:root:
 Dataframe -> Works Completed

INFO:root: Shape: 
(15001, 11)

INFO:root: Dataframe Datatypes 
sr_no                           object
work_category                   object
work                            object
state                           object
ida                             object
work_description                object
honble_members_of_parliament    object
constituency                    object
image                           object
completion_date                 object
amount_disbursed                object
dtype: object

INFO:root:

In [5]:
print(al.isna().sum()[al.isna().sum()>0])
print(wc.isna().sum()[wc.isna().sum()>0])
print(ws.isna().sum()[ws.isna().sum()>0])

allocated_amount    1
dtype: int64
work_description      55
image               5054
amount_disbursed       6
dtype: int64
work_description    38
dtype: int64


In [6]:
def numeric_conversion(df:pd.DataFrame, columns: list) -> pd.DataFrame:
    for cols in columns:
        df[cols] = pd.to_numeric(df[cols], errors="coerce", downcast="float")
    return df

al = numeric_conversion(al, ["allocated_amount"])
wc = numeric_conversion(wc, ["amount_disbursed"])
ws = numeric_conversion(ws, ["sanction_amount"])

In [7]:
print(al.isna().sum()[al.isna().sum()>0])
print(wc.isna().sum()[wc.isna().sum()>0])
print(ws.isna().sum()[ws.isna().sum()>0])

allocated_amount    2
dtype: int64
work_description      55
image               5054
amount_disbursed       7
dtype: int64
work_description    38
sanction_amount      1
dtype: int64


In [8]:
def replace_strings_with_real_nulls(df: pd.DataFrame) -> pd.DataFrame:
    df = df.replace("NaN", np.nan)
    return df
    
al = replace_strings_with_real_nulls(al)
wc = replace_strings_with_real_nulls(wc)
ws = replace_strings_with_real_nulls(ws)

In [9]:
def strip_whitespace(df: pd.DataFrame) -> pd.DataFrame:
    for col_name in df.columns:
        if df[col_name].dtype == "object":
            df[col_name] = df[col_name].str.strip()
            df[col_name] = df[col_name].str.replace(r"\t", " ", regex=True) 

    return df

al = strip_whitespace(al)
wc = strip_whitespace(wc)
ws = strip_whitespace(ws)

# al.style.format({"allocated_amount": "{:,.2f}"})
# wc.style.format({"amount_disbursed": "{:,.2f}"})
# wc.style.format({"sanction_amount": "{:,.2f}"})
# DO-NOT UNCOMMENT THESE!!

In [10]:
def convert_real_datetime(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    for cols in columns:
        initial_count =  df[cols].isna().sum()
        df[cols] = pd.to_datetime(df[cols], errors="coerce",  format="%d-%b-%Y")
        failed_count = df[cols].isna().sum()
        failed = failed_count - initial_count
        logging.info(f"Column '{cols}': {failed} unparseable entries turned into NaT.")
    return df

wc_datetime_list = ["completion_date"]
ws_datetime_list = ["recommended_date", "sanction_date"]
wc = convert_real_datetime(wc, wc_datetime_list)
ws = convert_real_datetime(ws, ws_datetime_list)

INFO:root:Column 'completion_date': 1 unparseable entries turned into NaT.
INFO:root:Column 'recommended_date': 1 unparseable entries turned into NaT.
INFO:root:Column 'sanction_date': 1 unparseable entries turned into NaT.


In [11]:
def drop_duplicate_rows(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = df.shape[0]
    df = df.drop_duplicates()
    after = df.shape[0]
    logging.info(f"{name}: dropped {before - after} duplicate rows")
    return df

al = drop_duplicate_rows(al, "Allocated Limit")
ws = drop_duplicate_rows(ws, "Works Sanctioned")
wc = drop_duplicate_rows(wc, "Works Completed")

INFO:root:Allocated Limit: dropped 0 duplicate rows
INFO:root:Works Sanctioned: dropped 0 duplicate rows
INFO:root:Works Completed: dropped 0 duplicate rows


In [12]:
al = al.drop(columns=["sr_no"])
ws = ws.drop(columns=["sr_no"])
wc = wc.drop(columns=["sr_no", "image"])

In [13]:
diagnose_dataframe(al, "Allocated Limit (cleaned)")
diagnose_dataframe(ws, "Works Sanctioned (cleaned)")
diagnose_dataframe(wc, "Works Completed (cleaned)")

INFO:root:
 Dataframe -> Allocated Limit (cleaned)

INFO:root: Shape: 
(544, 4)

INFO:root: Dataframe Datatypes 
state                             object
honble_members_of_parliaments     object
constituency                      object
allocated_amount                 float64
dtype: object

INFO:root: Dataframe null columns 
allocated_amount    2
dtype: int64

INFO:root: Datarframe duplicates 
0
INFO:root:
 Dataframe -> Works Sanctioned (cleaned)

INFO:root: Shape: 
(6001, 11)

INFO:root: Dataframe Datatypes 
work_category                           object
work                                    object
state                                   object
ida                                     object
honble_members_of_parliament            object
constituency                            object
work_description                        object
recommended_date                datetime64[ns]
sanction_date                   datetime64[ns]
sanction_amount                        float64
work_status    

,state,honble_members_of_parliaments,constituency,allocated_amount
433,Haryana,Shri Dharambir Singh,BHIWANI MAHENDRAGARH,1.470000e+08
434,West Bengal,Shri Abhishek Banerjee,DIAMOND HARBOUR,1.470000e+08
435,Punjab,Shri Amar Singh,FATEHGARH SAHIB(SC),1.470000e+08
436,Gujarat,Shri Amit Shah,GANDHINAGAR,1.470000e+08
437,Madhya Pradesh,Shri Anil Firojiya,UJJAIN(SC),1.470000e+08
438,Rajasthan,Shri Arjun Ram Meghwal,BIKANER(SC),1.470000e+08
439,Uttar Pradesh,Shri Arun Kumar Sagar,SHAHJAHANPUR(SC),1.470000e+08
440,Karnataka,Shri B Y Raghavendra,SHIMOGA,1.470000e+08
441,Kerala,Shri Benny Behanan,CHALAKUDY,1.646748e+08
442,Odisha,Shri Bhartruhari Mahtab,CUTTACK,1.470000e+08


In [15]:
wc

,work_category,work,state,ida,work_description,honble_members_of_parliament,constituency,completion_date,amount_disbursed
0,Normal/Others,WS/MP418/2024-2025/133409-Construction of road...,Bihar,ARARIA(DISTRICT PLANNING OFFICER ARARIA_IDA),PCC Road from Permeshwar Bhagat house to Ramde...,Pradeep Kumar Singh,ARARIA,2024-09-05,448127.0
1,Normal/Others,WS/MP18152/2024-2025/133691-Construction of ro...,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),2025-04-07,300000.0
2,Normal/Others,WS/MP345/2024-2025/134140-Construction of road...,Kerala,KOLLAM(DISTRICT COLLECTOR KOLLAM_IDA),Concreting of road from Janathavayanasala - P...,Shri NK Premachandran,KOLLAM,2024-08-12,293492.0
3,Normal/Others,WS/MP18152/2024-2025/133686-Construction of ro...,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),2025-04-07,300000.0
4,Normal/Others,WS/MP18152/2024-2025/133690-Construction of ro...,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),2025-04-07,300000.0
...,...,...,...,...,...,...,...,...,...
14996,Normal/Others,WS/MP18344/2025-2026/176584-Construction of ro...,Gujarat,RAJKOT(DISTRICT COLLECTOR RAJKOT_IDA),Paver block work from Bus Station to Ram Mandir,DR. MANSUKH MANDAVIYA,PORBANDAR,2025-09-13,400000.0
14997,Normal/Others,WS/MP797/2025-2026/218228-Installation of fixe...,Uttar Pradesh,FARRUKHABAD(DISTRICT MAGISTRAE FARRUKHABAD_IDA),Establishment of an open gym at a public place...,Mukesh Rajput,FARRUKHABAD,2025-12-03,822505.0
14998,Normal/Others,WS/MP574/2025-2026/219836-Purchase of smart bo...,Gujarat,PATAN(DISTRICT COLLECTOR PATAN_IDA),SMART CLASS WORK IN UNDRA PRIMARY SCHOOL,Bharatsinhji Shankarji Dabhi,PATAN,2025-09-08,498875.0
14999,Normal/Others,WS/MP797/2025-2026/218238-Installation of fixe...,Uttar Pradesh,FARRUKHABAD(DISTRICT MAGISTRAE FARRUKHABAD_IDA),Establishment of an open gym behind the Junior...,Mukesh Rajput,FARRUKHABAD,2025-12-03,822505.0


al

In [57]:
def mp_name_clean(raw_name: str) -> str:
    cleaned_name = re.sub(r"^(Shri|Smt|Dr\.|Er\.)\s*", "", raw_name).upper()
    return cleaned_name

# al2 = al.copy()
# al2["honble_members_of_parliaments"] = al2["honble_members_of_parliaments"].apply(normalize_mp_name)

al["honble_members_of_parliaments"] = al["honble_members_of_parliaments"].apply(mp_name_clean)
ws["honble_members_of_parliament"] = ws["honble_members_of_parliament"].apply(mp_name_clean)
wc["honble_members_of_parliament"] = wc["honble_members_of_parliament"].apply(mp_name_clean)

In [74]:
al_set = set(al["honble_members_of_parliaments"].unique())
ws_set = set(ws["honble_members_of_parliament"].unique())
wc_set = set(wc["honble_members_of_parliament"].unique())

ws_matches = ws_set.intersection(al_set)
wc_matches = wc_set.intersection(al_set)

sanctioned_match_rate = len(ws_matches) / len(ws_set) if ws_set else 0.0
completed_match_rate = len(wc_matches) / len(wc_set) if wc_set else 0.0

logging.info("--- Overlap Summary ---")
logging.info(f"Sanctioned: {len(ws_matches)} names match Allocated out of {len(ws_set)} total.")
logging.info(f"Sanctioned Match Rate: {sanctioned_match_rate:.2%}")

logging.info(f"Completed: {len(wc_matches)} names match Allocated out of {len(wc_set)} total.")
logging.info(f"Completed Match Rate: {completed_match_rate:.2%}")

if sanctioned_match_rate < 0.90:
    logging.warning(f"Low match rate in Sanctioned! Sample of unmatched names: {list(ws_set - al_set)[:15]}")

if completed_match_rate < 0.90:
    logging.warning(f"Low match rate in Completed! Sample of unmatched names: {list(wc_set - al_set)[:15]}")

INFO:root:--- Overlap Summary ---
INFO:root:Sanctioned: 212 names match Allocated out of 212 total.
INFO:root:Sanctioned Match Rate: 100.00%
INFO:root:Completed: 417 names match Allocated out of 417 total.
INFO:root:Completed Match Rate: 100.00%
